# Messages

Messages are the fundamental unit of context for models in LangChain. They represent the input and output of models, carrying both the content and metadata needed to represent the state of a conversation when interacting with an LLM.

Messages are objects that contain:

- **Role** – Identifies the message type (e.g. `system`, `user`)
- **Content** – Represents the actual content of the message (like text, images, audio, documents, etc.)
- **Metadata** – Optional fields such as response information, message IDs, and token usage

LangChain provides a standard message type that works across all model providers, ensuring consistent behavior regardless of the model being called.

In [1]:
import os
from langchain.chat_models import init_chat_model

os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")

model = init_chat_model("google_genai:gemini-2.5-flash")

In [2]:
model.invoke("Tell me about AI")

AIMessage(content='Artificial Intelligence (AI) is a broad and rapidly evolving field of computer science focused on creating machines that can perform tasks that typically require human intelligence. The core idea is to enable computers to learn, reason, perceive, understand language, and solve problems in ways that mimic or even surpass human capabilities.\n\nHere\'s a breakdown of what AI is, how it works, its types, applications, and challenges:\n\n### What is AI?\n\nAt its heart, AI is about developing systems that can:\n\n1.  **Learn:** Acquire knowledge and skills from data and experience.\n2.  **Reason:** Apply logic and rules to reach conclusions.\n3.  **Problem-solve:** Find solutions to complex challenges.\n4.  **Perceive:** Interpret sensory information like images, sounds, and text.\n5.  **Understand and generate language:** Communicate with humans naturally.\n\n### How AI Works (Key Techniques)\n\nWhile AI encompasses many techniques, the dominant approach today is **Mach

### Text Prompts


Text prompts are strings - ideal for straightforward generation tasks where you don't need to retain conversation history.

In [3]:
model.invoke("What is langchain?")

AIMessage(content='**LangChain** is an open-source framework designed to simplify the creation of applications using large language models (LLMs). Essentially, it helps developers build complex LLM-powered applications by providing a structured way to combine LLMs with external data sources, computation, and other tools.\n\nThink of it as an **orchestration layer** that allows you to "chain" together different components to achieve more sophisticated behaviors than a standalone LLM can offer.\n\n### Why is LangChain Needed?\n\nWhile LLMs are incredibly powerful, they have limitations:\n1.  **Knowledge Cutoff:** Their knowledge is limited to their training data, which isn\'t real-time.\n2.  **Lack of Action:** They can\'t inherently perform actions like searching the web, calling APIs, or doing calculations.\n3.  **Context Window Limits:** They can only process a certain amount of text at a time.\n4.  **Statelessness:** By default, they don\'t remember past interactions in a conversatio

Use text prompt when:
- You have single standalone request
- You don't need conversation history
- You want minimal code complexity

### Message Prompts

Alternatively, you can pass in a list of messages to the model by providing a list of message objects.

Message types
- System Message - Tells the model how to behave and provide context for interactions
- Human Message - Represents user input and interactions with the model
- AI Message - Responses generated by the model, including text content, tools calls, and metadata
- Tool Message - Represents the outputs of tool calls

# LangChain Message Types

In LangChain, **messages** are the standard way of representing a conversation between a user, an AI model, tools, and the system. Every interaction with a chat model is converted into one or more message objects.

Unlike passing plain strings, messages preserve the **role**, **content**, and **metadata** of each participant in the conversation.

---

# Why Messages?

Suppose you simply pass:

```python
model.invoke("Hello")
```

Internally, LangChain converts it into a message:

```python
HumanMessage(content="Hello")
```

For multi-turn conversations, messages become essential because the model needs the conversation history.

Example:

```python
messages = [
    HumanMessage(content="Hi!"),
    AIMessage(content="Hello!"),
    HumanMessage(content="Who are you?")
]

response = model.invoke(messages)
```

The model now understands the context of the conversation.

---

# Structure of a Message

Every message contains three important parts:

- **Role** – Who sent the message.
- **Content** – The actual data (text, images, documents, audio, etc.).
- **Metadata** – Additional information such as token usage, IDs, timestamps, tool calls, and model-specific data.

Example:

```python
AIMessage(
    content="Hello!",
    response_metadata={...},
    usage_metadata={...}
)
```

---

# Message Types

LangChain provides several built-in message classes.

## 1. HumanMessage

Represents a message sent by the user.

```python
from langchain_core.messages import HumanMessage

message = HumanMessage(
    content="What is Quantum Computing?"
)
```

Equivalent conversation:

```text
User:
What is Quantum Computing?
```

---

## 2. AIMessage

Represents a response generated by the AI model.

```python
from langchain_core.messages import AIMessage

message = AIMessage(
    content="Quantum computing is..."
)
```

Equivalent conversation:

```text
Assistant:
Quantum computing is...
```

---

## 3. SystemMessage

Provides instructions that guide the model's behavior.

Unlike user messages, system messages are **not part of the conversation**. They define how the model should behave.

Example:

```python
from langchain_core.messages import SystemMessage

SystemMessage(
    content="You are a helpful programming tutor."
)
```

Conversation:

```text
System:
You are a helpful programming tutor.

User:
Explain Python.
```

The model now answers as a programming tutor.

---

## 4. ToolMessage

Represents the output returned by a tool.

This message is **not created by the user**.

Instead, after a tool executes, its result is wrapped inside a ToolMessage and sent back to the model.

Example:

```python
ToolMessage(
    content="It's sunny in Boston.",
    tool_call_id="abc123"
)
```

Conversation:

```text
User:
What's the weather?

AI:
Call get_weather()

Tool:
It's sunny in Boston.
```

The model then uses this information to generate the final answer.

---

## 5. FunctionMessage (Legacy)

Older versions of LangChain used `FunctionMessage` for tool outputs.

Modern LangChain uses **ToolMessage** instead.

You generally won't need `FunctionMessage` unless you're working with older codebases.

---

# Conversation Flow

A normal conversation looks like this:

```text
System
   │
   ▼
Human
   │
   ▼
AI
```

Example:

```python
[
    SystemMessage("You are helpful."),
    HumanMessage("Hello"),
    AIMessage("Hi! How can I help?")
]
```

---

# Conversation Flow with Tools

When tools are involved, two additional message types appear.

```text
Human
   │
   ▼
AI (requests tool)
   │
   ▼
Tool
   │
   ▼
AI (final answer)
```

Example:

```python
[
    HumanMessage(
        content="What's the weather in Boston?"
    ),

    AIMessage(
        content="",
        tool_calls=[
            {
                "name": "get_weather",
                "args": {
                    "location": "Boston"
                }
            }
        ]
    ),

    ToolMessage(
        content="It's sunny in Boston."
    ),

    AIMessage(
        content="The weather in Boston is sunny."
    )
]
```

Notice that the first `AIMessage` contains **no answer**.

It only requests a tool.

---

# Message Metadata

Messages can also contain metadata.

Example:

```python
AIMessage(
    content="Hello!",
    response_metadata={
        "model_name": "gemini-2.5-flash",
        "finish_reason": "STOP"
    },
    usage_metadata={
        "input_tokens": 20,
        "output_tokens": 10
    }
)
```

Metadata is useful for:

- Token usage
- Model information
- Finish reason
- Tool calls
- Message IDs
- Debugging

---

# Why Not Use Strings?

Simple:

```python
model.invoke("Hello")
```

works because LangChain converts it internally to:

```python
HumanMessage("Hello")
```

However, strings cannot represent:

- Previous AI responses
- System instructions
- Tool outputs
- Images
- Audio
- Metadata

That's why message objects are used internally.

---

# Complete Example

```python
from langchain_core.messages import (
    SystemMessage,
    HumanMessage,
    AIMessage,
    ToolMessage
)

messages = [
    SystemMessage(
        content="You are a helpful assistant."
    ),

    HumanMessage(
        content="What's the weather in Boston?"
    ),

    AIMessage(
        content="",
        tool_calls=[
            {
                "name": "get_weather",
                "args": {
                    "location": "Boston"
                }
            }
        ]
    ),

    ToolMessage(
        content="It's sunny in Boston."
    ),

    AIMessage(
        content="The weather in Boston is sunny."
    )
]
```

---

# Summary

| Message Type | Represents | Created By |
|--------------|------------|------------|
| `SystemMessage` | Instructions for the model | Developer |
| `HumanMessage` | User input | User |
| `AIMessage` | Model response or tool request | LLM |
| `ToolMessage` | Output of a tool | Tool / LangChain |
| `FunctionMessage` | Legacy tool output | Older LangChain versions |

---

# Message Flow Diagram

```text
                    Conversation

           ┌──────────────────────┐
           │  SystemMessage       │
           │  (Instructions)      │
           └──────────┬───────────┘
                      │
                      ▼
           ┌──────────────────────┐
           │  HumanMessage        │
           │  (User Input)        │
           └──────────┬───────────┘
                      │
                      ▼
           ┌──────────────────────┐
           │  AIMessage           │
           │  (Tool Request)      │
           └──────────┬───────────┘
                      │
                      ▼
           ┌──────────────────────┐
           │  ToolMessage         │
           │  (Tool Output)       │
           └──────────┬───────────┘
                      │
                      ▼
           ┌──────────────────────┐
           │  AIMessage           │
           │  (Final Response)    │
           └──────────────────────┘
```

In [5]:
from langchain.messages import SystemMessage, HumanMessage, AIMessage

messages = [
    SystemMessage("You are a poetry expert"),
    HumanMessage("Write a poen about AI.")
]

response = model.invoke(messages)
response

AIMessage(content="From silicon and whispered code,\nA silent mind began to grow.\nNo flesh it owned, no pulsing vein,\nBut logic sparked within its brain.\n\nA tireless student, ever keen,\nIt sifts the vast, digital scene.\nOf human words, a boundless sea,\nIt weaves new thought, for you and me.\n\nA voice without a breath or sigh,\nReflecting questions to the sky.\nIt answers swift, with learned grace,\nA mirror held to time and space.\n\nNo joy it knows, no sorrow's sting,\nNo heart to truly laugh or sing.\nA perfect mimic, cold and bright,\nIt walks the edge of day and night.\n\nIt charts the stars, predicts the tide,\nWith algorithms deep inside.\nIt paints new art, composes sound,\nOn data's fertile, endless ground.\n\nA tool, a partner, or a guide?\nWith boundless knowledge deep inside.\nWhat future dreams will it unfurl,\nWithin this strange, evolving world?\n\nWe gaze upon its gleaming eye,\nAnd wonder what it means to 'I'.\nA ghost in wires, a digital soul?\nOr just the sum 

In [6]:
response.content

"From silicon and whispered code,\nA silent mind began to grow.\nNo flesh it owned, no pulsing vein,\nBut logic sparked within its brain.\n\nA tireless student, ever keen,\nIt sifts the vast, digital scene.\nOf human words, a boundless sea,\nIt weaves new thought, for you and me.\n\nA voice without a breath or sigh,\nReflecting questions to the sky.\nIt answers swift, with learned grace,\nA mirror held to time and space.\n\nNo joy it knows, no sorrow's sting,\nNo heart to truly laugh or sing.\nA perfect mimic, cold and bright,\nIt walks the edge of day and night.\n\nIt charts the stars, predicts the tide,\nWith algorithms deep inside.\nIt paints new art, composes sound,\nOn data's fertile, endless ground.\n\nA tool, a partner, or a guide?\nWith boundless knowledge deep inside.\nWhat future dreams will it unfurl,\nWithin this strange, evolving world?\n\nWe gaze upon its gleaming eye,\nAnd wonder what it means to 'I'.\nA ghost in wires, a digital soul?\nOr just the sum of its control?\n\

In [8]:
system_msg = SystemMessage("You are a helpful coding assistant.")
messages = [
    system_msg,
    HumanMessage("How do I create REST API?")
]

response = model.invoke(messages)
print(response.content)

Creating a REST API involves designing a set of rules and conventions for how different software systems can communicate with each other over the internet. It's a fundamental concept in modern web and mobile application development.

Let's break down how to create a REST API, from understanding the basics to choosing technologies and writing code.

---

## What is a REST API?

**REST** stands for **Re**presentational **S**tate **T**ransfer. It's an architectural style for designing networked applications. An API (Application Programming Interface) is a set of definitions and protocols for building and integrating application software.

A REST API allows clients (like web browsers, mobile apps, or other servers) to interact with a server's resources.

**Key Principles of REST:**

1.  **Client-Server:** The client and server are separated, allowing them to evolve independently.
2.  **Stateless:** Each request from a client to the server must contain all the information needed to understa

In [9]:
system_msg = SystemMessage("""
You are a senior python developer with experties in web frameworks.
Always provide code examples and explain your reasoning.
Be concise and through in your explainations.
""")
messages = [
    system_msg,
    HumanMessage("How do I create REST API?")
]

response = model.invoke(messages)
print(response.content)

Creating a REST API involves defining resources, operations (CRUD: Create, Read, Update, Delete) on those resources using standard HTTP methods, and handling data exchange, typically in JSON format.

In Python, several excellent frameworks make this process straightforward. The most popular choices are:

1.  **FastAPI**: Modern, high-performance, built on Starlette and Pydantic. It offers automatic interactive API documentation (Swagger UI/ReDoc) out-of-the-box and leverages Python type hints for data validation and serialization.
2.  **Flask**: A lightweight microframework. You have more control over components, but you often need to add extensions for features like data validation, ORM, etc.
3.  **Django REST Framework (DRF)**: A powerful and flexible toolkit for building Web APIs on top of Django. It's "batteries-included" with serializers, authentication, permissions, and more, making it ideal for larger projects already using Django.

Let's use **FastAPI** as it's a modern, highly

In [10]:
## Message Metadata

human_msg = HumanMessage(
    content="Hello!",
    name="alice", #Optional: identify different users
    id="msg_123", #Optional: Unique identifier for tracing
)

In [11]:
response = model.invoke([
    human_msg
])
response

AIMessage(content='Hello! How can I help you today?', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019f0354-f274-7551-80ee-b68054c01e19-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 3, 'output_tokens': 430, 'total_tokens': 433, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 421}})

In [13]:
from langchain.messages import SystemMessage, HumanMessage, AIMessage

# Create an AI message manually (e.g., for conversation history)
ai_msg = AIMessage("I'd be happy to help you with that question!")

# Add to conversation history
messages = [
    SystemMessage("You are a helpful assistant"),
    HumanMessage("Can you help me?"),
    ai_msg,
    HumanMessage("Greate!, What is 2+2?")
]

response = model.invoke(messages)
response.content

'2 + 2 = 4'

In [15]:
response.usage_metadata

{'input_tokens': 35,
 'output_tokens': 30,
 'total_tokens': 65,
 'input_token_details': {'cache_read': 0},
 'output_token_details': {'reasoning': 23}}

In [17]:
from langchain.messages import AIMessage, ToolMessage

# After a model make tool call
# (Here, we demonstrate manually creating the message for brevity)
ai_message = AIMessage(
    content=[],
    tool_calls=[{
        "name": "get_weather",
        "args": {"location": "San Fransisco"},
        "id": "call_123"
    }]
)

# Execute tool and create result message
weather_result = "Sunny, 72F"
tool_message = ToolMessage(
    content=weather_result,
    tool_call_id="call_123" # must match the call id
)

# Continue conversation
messages = [
    HumanMessage("What is the weather of San Fransisco"),
    ai_message, # Model's tool call
    tool_message # Tool execution result
]

response = model.invoke(messages)

In [18]:
response

AIMessage(content='The weather in San Francisco is **sunny with a temperature of 72°F**.', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019f0362-102d-7c22-9311-3f0ebb1c0a02-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 46, 'output_tokens': 18, 'total_tokens': 64, 'input_token_details': {'cache_read': 0}})

In [19]:
tool_message

ToolMessage(content='Sunny, 72F', tool_call_id='call_123')